# Laboratorio 13 — Ordenar sin particionar

**Duración: 45 minutos.**

En el laboratorio 04 aprendiste a particionar, y funcionó: de treinta mil filas leídas a
dos mil quinientas. Pero particionar no es gratis ni siempre se puede, y hoy vas a ver que
**no es la única palanca**.

La pregunta de hoy:

> Si no puedo particionar esta tabla, **¿estoy condenado a leerla entera?**

---

### En el almacén

Particionar es poner **pasillos rotulados**: todo lo de enero en un pasillo, todo lo de
febrero en otro. Para eso hay que elegir el rótulo, y a veces no hay uno bueno.

Ordenar es otra cosa. Es dejar las cajas **en orden de fecha** en un solo pasillo, sin
rótulos. Si alguien pregunta por junio, no hace falta abrir todas: con mirar por fuera la
primera y la última caja de cada montón, se sabe cuáles no pueden contener junio.

Eso funciona **si hay varios montones**. Con una sola caja gigante, ordenar por dentro no
sirve de nada. Ese es el primer descubrimiento de hoy.


## Paso 0 — Una tabla ordenada, sin particiones

Primero, a tu espacio de trabajo.

In [1]:
%%sql
-- Celda 0.1
USE mi_espacio

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/24 20:51:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Partimos de cero.

In [2]:
%%sql
-- Celda 0.2
DROP TABLE IF EXISTS documentos_lab13

Ahora la tabla del día. Fíjate bien en lo que tiene y en lo que no: lleva
`ORDER BY fecha_emision`, **pero no lleva `PARTITIONED BY`**. Un año completo de
documentos, ordenados por fecha, en una tabla sin particiones.

In [3]:
%%sql
-- Celda 0.3
CREATE TABLE documentos_lab13 USING iceberg AS
SELECT * FROM curso.dte_2024
ORDER BY fecha_emision

¿Cuántos quedaron?

In [4]:
%%sql
-- Celda 0.4
SELECT count(*) AS documentos FROM documentos_lab13

documentos
30000


Treinta mil, los de siempre. Y ahora la celda que decide el laboratorio: **en cuántos
archivos quedaron**, y qué rango de fechas abarca cada uno.

In [5]:
%%sql
-- Celda 0.5
SELECT record_count                                AS filas,
       readable_metrics.fecha_emision.lower_bound AS desde,
       readable_metrics.fecha_emision.upper_bound AS hasta
FROM mi_espacio.documentos_lab13.files
ORDER BY desde

filas,desde,hasta
30000,2024-01-01,2024-12-31


## Paso 1 — Partirla en varios archivos

**Un solo archivo, del 1 de enero al 31 de diciembre.**

Ahí está el problema, y es importante entenderlo antes de seguir: la tabla **está**
ordenada por dentro, pero eso no le sirve a nadie. El motor decide qué leer mirando
archivos enteros, y solo hay uno. Para responder por junio tiene que abrirlo, y con él
entra el año completo.

Ordenar sin tener varios archivos no hace nada. Vamos a partirla.

Para eso usamos el mismo `rewrite_data_files` del laboratorio 08, pero pidiéndole lo
contrario: allá juntaba archivos chicos en uno grande, aquí le decimos que apunte a
archivos de unos 250 kilobytes y que ordene por fecha al reescribir.

In [6]:
%%sql
-- Celda 1.1
CALL spark_catalog.system.rewrite_data_files(
    table      => 'mi_espacio.documentos_lab13',
    strategy   => 'sort',
    sort_order => 'fecha_emision ASC NULLS LAST',
    options    => map('target-file-size-bytes', '250000',
                      'min-input-files',        '1',
                      'rewrite-all',            'true')
)

rewritten_data_files_count,added_data_files_count,rewritten_bytes_count
1,9,720233


Y ahora mira otra vez los archivos y sus rangos.

In [7]:
%%sql
-- Celda 1.2
SELECT record_count                                AS filas,
       readable_metrics.fecha_emision.lower_bound AS desde,
       readable_metrics.fecha_emision.upper_bound AS hasta
FROM mi_espacio.documentos_lab13.files
ORDER BY desde

filas,desde,hasta
4000,2024-01-01,2024-02-18
4000,2024-02-18,2024-04-06
1791,2024-04-06,2024-04-27
4000,2024-04-28,2024-06-16
4000,2024-06-16,2024-08-04
1807,2024-08-04,2024-08-26
4000,2024-08-27,2024-10-14
4000,2024-10-14,2024-12-02
2402,2024-12-02,2024-12-31


## Paso 2 — La regla de medir

**Nueve archivos, cada uno con su tramo del año**, del más antiguo al más nuevo y sin
pisarse salvo en el día del borde. Eso es lo que hace el orden: agrupa por **rango**.

Ahora la regla de medir del laboratorio 04, la misma consulta, sobre esta tabla. Cuenta
cuántos archivos tendría que abrir el motor para responder por junio.

In [8]:
%%sql
-- Celda 2.1
SELECT count(*)          AS archivos_a_leer,
       sum(record_count)  AS filas_a_leer
FROM mi_espacio.documentos_lab13.files
WHERE readable_metrics.fecha_emision.upper_bound >= DATE '2024-06-01'
  AND readable_metrics.fecha_emision.lower_bound <= DATE '2024-06-30'

archivos_a_leer,filas_a_leer
2,8000


## Paso 3 — El cuadro completo

**Dos archivos y unas ocho mil filas**, contra los treinta mil de antes. Sin una sola
partición.

Este es el cuadro que cierra la primera mitad del laboratorio, y conviene copiarlo tal
cual:

| Tabla | Archivos a leer | Filas a leer |
|---|---|---|
| Sin nada, un archivo | 1 | 30.000 |
| **Ordenada por fecha, nueve archivos** | **2** | **~8.000** |
| Particionada por mes, laboratorio 04 | 1 | 2.500 |

Léelo de arriba abajo. Ordenar no llega a lo que consigue particionar, y era esperable:
un rango de fechas es más grueso que un rótulo exacto. Pero pasa de leer el año entero a
leer dos meses, **sin declarar ninguna partición**.

Y eso importa cuando particionar no es una opción: cuando no hay una columna con pocos
valores distintos, cuando el criterio de consulta cambia seguido, o cuando particionar
generaría tantos archivos chicos que el remedio sería peor.

## Paso 4 — Lo que pasa cuando llega una carga desordenada

Hasta aquí todo bien. Pero el orden es frágil, y esto es lo que hay que saber antes de
apoyarse en él.

Llega una carga nueva. Los mismos documentos del año, pero **en orden aleatorio**, como
llegaría de un sistema que no ordena nada.

Fíjate en que es **el mismo año otra vez**: la tabla va a quedar con sesenta mil documentos
y cada uno repetido dos veces. Es a propósito, y no importa para lo de hoy, porque lo que
vamos a mirar es **cómo quedan repartidos los archivos**, no qué dice cada fila. Lo único
que necesitamos de esta carga es que llegue desordenada.

In [9]:
%%sql
-- Celda 4.1
INSERT INTO documentos_lab13
SELECT * FROM curso.dte_2024 ORDER BY rand()

Mira los archivos otra vez.

In [10]:
%%sql
-- Celda 4.2
SELECT record_count                                AS filas,
       readable_metrics.fecha_emision.lower_bound AS desde,
       readable_metrics.fecha_emision.upper_bound AS hasta
FROM mi_espacio.documentos_lab13.files
ORDER BY desde

filas,desde,hasta
30000,2024-01-01,2024-12-31
4000,2024-01-01,2024-02-18
4000,2024-02-18,2024-04-06
1791,2024-04-06,2024-04-27
4000,2024-04-28,2024-06-16
4000,2024-06-16,2024-08-04
1807,2024-08-04,2024-08-26
4000,2024-08-27,2024-10-14
4000,2024-10-14,2024-12-02
2402,2024-12-02,2024-12-31


**Ahí está el daño.** Hay un archivo nuevo que abarca del 1 de enero al 31 de diciembre,
porque sus filas llegaron mezcladas. Ese archivo va a entrar en **todas** las consultas,
pregunten por el mes que pregunten.

Vuelve a medir para junio.

In [11]:
%%sql
-- Celda 4.3
SELECT count(*)          AS archivos_a_leer,
       sum(record_count)  AS filas_a_leer
FROM mi_espacio.documentos_lab13.files
WHERE readable_metrics.fecha_emision.upper_bound >= DATE '2024-06-01'
  AND readable_metrics.fecha_emision.lower_bound <= DATE '2024-06-30'

archivos_a_leer,filas_a_leer
3,38000


## Paso 5 — Reordenar lo que ya está

Subió, y subió mucho: el archivo desordenado se suma a cualquier consulta.

La buena noticia es que esto se arregla, y con la misma herramienta del paso 1. No hay que
recargar nada ni recrear la tabla: se reordena **lo que ya está escrito**.

In [12]:
%%sql
-- Celda 5.1
CALL spark_catalog.system.rewrite_data_files(
    table      => 'mi_espacio.documentos_lab13',
    strategy   => 'sort',
    sort_order => 'fecha_emision ASC NULLS LAST',
    options    => map('target-file-size-bytes', '250000',
                      'min-input-files',        '1',
                      'rewrite-all',            'true')
)

rewritten_data_files_count,added_data_files_count,rewritten_bytes_count
10,18,1607712


Los archivos, por tercera vez.

In [13]:
%%sql
-- Celda 5.2
SELECT record_count                                AS filas,
       readable_metrics.fecha_emision.lower_bound AS desde,
       readable_metrics.fecha_emision.upper_bound AS hasta
FROM mi_espacio.documentos_lab13.files
ORDER BY desde

filas,desde,hasta
4000,2024-01-01,2024-01-25
4000,2024-01-25,2024-02-18
1476,2024-02-18,2024-02-26
4000,2024-02-27,2024-03-22
4000,2024-03-22,2024-04-15
1626,2024-04-15,2024-04-24
4000,2024-04-25,2024-05-20
4000,2024-05-20,2024-06-13
2734,2024-06-13,2024-06-29
4000,2024-06-30,2024-07-24


**Otra vez rangos ordenados**, ahora con sesenta mil documentos repartidos. Mide de
nuevo y compara con lo que tenías después del paso 4.

In [14]:
%%sql
-- Celda 5.3
SELECT count(*)          AS archivos_a_leer,
       sum(record_count)  AS filas_a_leer
FROM mi_espacio.documentos_lab13.files
WHERE readable_metrics.fecha_emision.upper_bound >= DATE '2024-06-01'
  AND readable_metrics.fecha_emision.lower_bound <= DATE '2024-06-30'

archivos_a_leer,filas_a_leer
3,10734


Y una última mirada a la libreta, para ver qué dejó escrito la reorganización.

In [15]:
%%sql
-- Celda 5.4
SELECT snapshot_id, committed_at, operation
FROM mi_espacio.documentos_lab13.snapshots
ORDER BY committed_at

snapshot_id,committed_at,operation
409318531158205441,2026-09-24 20:51:22.025,append
2466221423714303085,2026-09-24 20:51:24.761,replace
8953813324453195842,2026-09-24 20:51:26.055,append
9159148294110272846,2026-09-24 20:51:27.856,replace


## Cierre

Dos palancas para lo mismo, y no son excluyentes:

| | Agrupa por | Cuándo sirve |
|---|---|---|
| **Particionar** | valor exacto, `mes = 2024-06` | hay una columna con pocos valores distintos y estable |
| **Ordenar** | rango, `del 2 de mayo al 19 de junio` | no hay buen criterio de partición, o cambia |

Las dos hacen lo mismo en el fondo: que el motor **pueda descartar archivos sin abrirlos**.
Ese es el mecanismo, y es el mismo que viste en el laboratorio 04 con los rótulos.

Tres cosas que te llevas:

- **Ordenar sin varios archivos no hace nada.** Con un archivo gigante da igual lo ordenado
  que esté por dentro.
- **El orden se degrada solo.** Cada carga desordenada agrega un archivo que abarca todo y
  se mete en todas las consultas. No es un error, es cómo funciona.
- **Se repara sin recargar.** `rewrite_data_files` con `sort_order` reordena lo escrito, y
  es una rutina de mantenimiento como las del laboratorio 08.

Y se combinan. Una tabla puede estar particionada por mes **y** ordenada por RUT dentro de
cada partición. Ahí el motor descarta por mes y después descarta por rango, y lee todavía
menos.
